# Vocal Coach — Sprint 1 Demo

Visualizes the four artifacts produced by `scripts/run_pipeline.py` on a single GTSinger sample:

1. **Reference notes** (from GTSinger's per-segment annotation) as a piano-roll-style grid of MIDI pitches.
2. **NanoPitch F0** (the user's actual sung pitch, decoded with the realtime Viterbi decoder).
3. **Loudness** (per-frame RMS in dBFS).
4. **STARS phoneme bands** colored by detected vocal techniques.

Sprint 1 deliberately stops short of any alignment math — the demo is just the four tracks side-by-side on a shared time axis.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'vocal_coach').is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

from vocal_coach.schemas import Timeline

SAMPLE_DIR = ROOT / 'data' / 'samples' / 'EN-Alto-1__innocence__0000'
TIMELINE_PATH = SAMPLE_DIR / 'timeline.json'
assert TIMELINE_PATH.is_file(), f'Run scripts/run_pipeline.py first; missing {TIMELINE_PATH}'

timeline = Timeline.model_validate_json(TIMELINE_PATH.read_text(encoding='utf-8'))
print(f'sample_id : {timeline.sample_id}')
print(f'duration  : {timeline.duration_s:.2f} s')
print(f'reference : {len(timeline.reference.notes)} notes, {len(timeline.reference.phones)} phonemes, {len(timeline.reference.words)} words')
print(f'pitch     : {len(timeline.pitch.frames)} frames @ {timeline.pitch.hop_seconds*1000:.0f} ms hop')
print(f'loudness  : {len(timeline.loudness.frames)} frames')
if timeline.stars is not None:
    print(f'stars     : {len(timeline.stars.phonemes)} phonemes, {len(timeline.stars.notes)} notes; style={timeline.stars.style}')
else:
    print('stars     : not run / unavailable')

## Track extraction helpers

Convert each of the typed schema arrays into plain numpy vectors aligned by time. We also compute cents deviation against the nearest reference note for visualization, but no aggregation is done at the note level (that's sprint 2).

In [ ]:
ref = timeline.reference
pitch = timeline.pitch
loud = timeline.loudness
stars = timeline.stars

pitch_t = np.array([f.time for f in pitch.frames])
pitch_f0 = np.array([f.f0_hz for f in pitch.frames])
pitch_voicing = np.array([f.voicing_confidence for f in pitch.frames])

loud_t = np.array([f.time for f in loud.frames])
loud_db = np.array([f.rms_db for f in loud.frames])

ref_notes = ref.notes
all_midis = [n.midi_pitch for n in ref_notes]
midi_lo = min(all_midis) - 2 if all_midis else 50
midi_hi = max(all_midis) + 2 if all_midis else 70

def hz_to_midi_float(hz):
    out = np.full_like(hz, np.nan, dtype=np.float64)
    voiced = hz > 0
    out[voiced] = 69.0 + 12.0 * np.log2(hz[voiced] / 440.0)
    return out

pitch_midi = hz_to_midi_float(pitch_f0)
print(f'reference midi range: {midi_lo} - {midi_hi}')
print(f'voiced pitch frames: {int((pitch_f0 > 0).sum())} / {len(pitch_f0)}')

## Figure 1 — the four-track timeline

Stacked panels:
* Top: reference note grid + sung F0 in MIDI space.
* Middle: loudness (dBFS).
* Bottom: STARS phoneme bands, color-coded by detected technique.

In [ ]:
TECH_COLORS = {
    'mixed':      '#7B9EFF',
    'falsetto':   '#9C27B0',
    'breathe':    '#4CAF50',
    'pharyngeal': '#FF9800',
    'glissando':  '#F44336',
    'vibrato':    '#E91E63',
    'bubble':     '#00BCD4',
    'weak':       '#9E9E9E',
    'strong':     '#FFC107',
}
TECH_PRIORITY = ['vibrato', 'glissando', 'falsetto', 'pharyngeal', 'breathe', 'bubble', 'strong', 'weak', 'mixed']

fig, axes = plt.subplots(3, 1, figsize=(13, 8.5), sharex=True,
                         gridspec_kw={'height_ratios': [3, 1, 1.5]})
ax_pitch, ax_loud, ax_phon = axes

# --- reference notes as semi-transparent rectangles ---
for n in ref_notes:
    ax_pitch.add_patch(Rectangle((n.start_s, n.midi_pitch - 0.45),
                                 n.end_s - n.start_s, 0.9,
                                 facecolor='#a8c8ff', edgecolor='#3a5fcd', alpha=0.55, zorder=1))
    ax_pitch.text(0.5*(n.start_s+n.end_s), n.midi_pitch, n.lyric_word,
                  ha='center', va='center', fontsize=8, color='#1c2e66', zorder=2)

# --- user F0 in MIDI space ---
ax_pitch.plot(pitch_t, pitch_midi, color='#d62728', lw=1.4, label='NanoPitch F0', zorder=3)
ax_pitch.set_ylabel('MIDI pitch')
ax_pitch.set_ylim(midi_lo, midi_hi)
ax_pitch.set_yticks(range(midi_lo, midi_hi + 1))
ax_pitch.set_title(f'Reference notes (blue) and user F0 (red) — {timeline.sample_id}')
ax_pitch.grid(axis='y', linestyle=':', alpha=0.4)
ax_pitch.legend(loc='upper right')

# --- loudness panel ---
ax_loud.plot(loud_t, loud_db, color='#2ca02c', lw=1.0)
ax_loud.set_ylabel('Loudness (dBFS)')
ax_loud.grid(axis='y', linestyle=':', alpha=0.4)

# --- STARS phoneme bands ---
if stars is not None:
    n_phon = len(stars.phonemes)
    for ph in stars.phonemes:
        active_techs = [t for t in TECH_PRIORITY if ph.techniques.get(t, 0)]
        color = TECH_COLORS[active_techs[0]] if active_techs else '#dddddd'
        ax_phon.add_patch(Rectangle((ph.start_s, 0.0), ph.end_s - ph.start_s, 1.0,
                                     facecolor=color, edgecolor='white', alpha=0.85, lw=0.5))
        ax_phon.text(0.5*(ph.start_s+ph.end_s), 0.5, ph.phoneme,
                     ha='center', va='center', fontsize=7, color='black')
    legend_handles = [Rectangle((0,0),1,1, facecolor=TECH_COLORS[t]) for t in TECH_PRIORITY]
    ax_phon.legend(legend_handles, TECH_PRIORITY, loc='upper right', ncol=3, fontsize=7)
    ax_phon.set_ylim(0, 1)
    ax_phon.set_yticks([])
    ax_phon.set_ylabel('STARS phonemes')
else:
    ax_phon.text(0.5, 0.5, 'STARS output not available', ha='center', va='center', transform=ax_phon.transAxes)
    ax_phon.set_yticks([])

ax_phon.set_xlim(0, timeline.duration_s)
ax_phon.set_xlabel('time (s)')
fig.tight_layout()
plt.show()

## Figure 2 — cents deviation against a per-frame reference note

For every frame whose time falls inside a reference note window, compute the cents difference between the user's F0 and the reference's MIDI pitch. This is the simplest possible "how flat / how sharp am I" view. Sprint 2 turns this into per-note aggregates and tags.

In [ ]:
ref_midi_per_frame = np.full_like(pitch_t, np.nan, dtype=np.float64)
for n in ref_notes:
    inside = (pitch_t >= n.start_s) & (pitch_t < n.end_s)
    ref_midi_per_frame[inside] = n.midi_pitch

cents = 1200.0 * (pitch_midi - ref_midi_per_frame) / 12.0  # MIDI-distance * 100 cents/semitone
valid = (~np.isnan(cents)) & (~np.isnan(pitch_midi))
if valid.any():
    print(f'mean abs cents err (voiced+inside-note): {np.nanmean(np.abs(cents[valid])):.1f}')
    print(f'median signed cents (voiced+inside-note): {np.nanmedian(cents[valid]):+.1f}')

fig, ax = plt.subplots(figsize=(13, 3.5))
ax.axhline(0, color='gray', lw=0.7)
ax.axhspan(-50, 50, color='#cce5ff', alpha=0.5, label='within 50 cents (RPA)')
ax.plot(pitch_t[valid], cents[valid], color='#d62728', lw=1.0)
ax.set_xlim(0, timeline.duration_s)
ax.set_ylim(-200, 200)
ax.set_xlabel('time (s)')
ax.set_ylabel('cents deviation')
ax.set_title('User F0 vs nearest reference note (cents). Sprint 2 will aggregate this per-note.')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()